# Data Cleaning — US Elections Dashboard
Limpieza, imputación y normalización de los 5 datasets antes de cargar a PostgreSQL.

In [ ]:
import pandas as pd
import numpy as np
import re

BASE = 'backend/datasets'
OUT  = 'backend/datasets/cleaned'

import os
os.makedirs(OUT, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

---
## 1. Elections data

In [ ]:
el = pd.read_csv(f'{BASE}/elections_data.csv', encoding='utf-8-sig', dtype={'FIPS': str})
print(el.shape)
el.info()

In [ ]:
# --- FIPS: zero-pad a 5 caracteres ---
el['FIPS'] = el['FIPS'].str.strip().str.zfill(5)

# --- Drop columna OBJECTID (ID interno GIS, no semántico) ---
el.drop(columns=['OBJECTID'], inplace=True)

# --- Estandarizar nombres a snake_case ---
el.columns = [
    'county_name', 'state_name', 'state_abbr', 'fips',
    'votes_tot', 'votes_trump', 'votes_harris', 'votes_stein',
    'pct_trump', 'pct_harris', 'pct_stein',
    'winner_2024', 'winner_2020', 'winner_2016'
]

print('FIPS lengths:', el['fips'].str.len().unique())

In [ ]:
# --- Identificar la fila problemática (1 fila sin votos) ---
missing_row = el[el['votes_tot'].isna()]
print('Fila con votos nulos:')
print(missing_row[['county_name','state_name','fips','votes_tot','winner_2024']])

In [ ]:
# --- Porcentajes: el CSV los tiene en escala 0-1, la BD los necesita 0-100 ---
for col in ['pct_trump', 'pct_harris', 'pct_stein']:
    el[col] = el[col] * 100

print('Rango pct_trump:', el['pct_trump'].min(), '-', el['pct_trump'].max())

In [ ]:
# --- Stein: candidata de nicho, NaN = 0 votos (no balota en ese condado) ---
el['votes_stein'] = el['votes_stein'].fillna(0).astype('Int64')
el['pct_stein']   = el['pct_stein'].fillna(0.0)

# --- Fila con votos nulos: imputar votos con suma de candidatos ---
mask_null = el['votes_tot'].isna()
el.loc[mask_null, 'votes_trump']  = el.loc[mask_null, 'votes_trump'].fillna(0)
el.loc[mask_null, 'votes_harris'] = el.loc[mask_null, 'votes_harris'].fillna(0)
el.loc[mask_null, 'votes_tot']    = (
    el.loc[mask_null, 'votes_trump'] +
    el.loc[mask_null, 'votes_harris'] +
    el.loc[mask_null, 'votes_stein']
)
# Recalcular pct si es posible
for cand, pct in [('votes_trump','pct_trump'),('votes_harris','pct_harris'),('votes_stein','pct_stein')]:
    el.loc[mask_null & (el['votes_tot'] > 0), pct] = (
        el.loc[mask_null & (el['votes_tot'] > 0), cand] /
        el.loc[mask_null & (el['votes_tot'] > 0), 'votes_tot'] * 100
    )

# Imputar winner_2024 de la fila nula con el mayor pct
def infer_winner(row):
    if pd.isna(row['winner_2024']):
        cands = {'Trump': row['pct_trump'], 'Harris': row['pct_harris'], 'Stein': row['pct_stein']}
        return max(cands, key=cands.get) if any(v > 0 for v in cands.values()) else 'Unknown'
    return row['winner_2024']
el['winner_2024'] = el.apply(infer_winner, axis=1)

# winner_2020 / 2016: condados sin datos son típicamente Alaska (reportan a nivel borough)
el['winner_2020'] = el['winner_2020'].fillna('Unknown')
el['winner_2016'] = el['winner_2016'].fillna('Unknown')

print('Nulls restantes:')
print(el.isnull().sum()[el.isnull().sum() > 0])

In [ ]:
# --- Tipos finales ---
for c in ['votes_tot', 'votes_trump', 'votes_harris']:
    el[c] = pd.to_numeric(el[c], errors='coerce').fillna(0).astype('Int64')

# --- Derivar columnas de análisis útiles ---
el['margin_votes'] = (el['votes_trump'] - el['votes_harris']).abs()
el['margin_pct']   = (el['pct_trump']   - el['pct_harris']).abs().round(4)
el['competitiveness_score'] = (100 - el['margin_pct']).round(4)

# Filas con 0 votos totales: pct y margin quedan indefinidos -> 0.0 (no hubo elección reportada)
for c in ['pct_trump','pct_harris','pct_stein','margin_pct','competitiveness_score']:
    el[c] = el[c].fillna(0.0)

print('Nulls restantes:', el.isnull().sum().sum())
el.head(3)

In [ ]:
el.to_csv(f'{OUT}/elections_clean.csv', index=False)
print('Guardado elections_clean.csv -', el.shape)

---
## 2. Demographics data

In [ ]:
dem = pd.read_csv(f'{BASE}/DemographicsData.csv', encoding='utf-8-sig')
print(dem.shape)
dem.head(2)

In [ ]:
# --- Renombrar a snake_case compacto ---
rename_map = {
    'County': 'county_name',
    'State':  'state_name',
    'Age.Percent 65 and Older':                         'age_pct_65_older',
    'Age.Percent Under 18 Years':                       'age_pct_under_18',
    'Age.Percent Under 5 Years':                        'age_pct_under_5',
    "Education.Bachelor's Degree or Higher":            'edu_bachelors_pct',
    'Education.High School or Higher':                  'edu_hs_or_higher_pct',
    'Employment.Nonemployer Establishments':            'employment_nonemployer_estab',
    'Ethnicities.American Indian and Alaska Native Alone': 'ethnicity_native_pct',
    'Ethnicities.Asian Alone':                          'ethnicity_asian_pct',
    'Ethnicities.Black Alone':                          'ethnicity_black_pct',
    'Ethnicities.Hispanic or Latino':                   'ethnicity_hispanic_pct',
    'Ethnicities.Native Hawaiian and Other Pacific Islander Alone': 'ethnicity_pacific_islander_pct',
    'Ethnicities.Two or More Races':                    'ethnicity_two_or_more_pct',
    'Ethnicities.White Alone':                          'ethnicity_white_pct',
    'Ethnicities.White Alone\t not Hispanic or Latino': 'ethnicity_white_nonhispanic_pct',
    'Housing.Homeownership Rate':                       'housing_homeownership_pct',
    'Housing.Households':                               'housing_households',
    'Housing.Housing Units':                            'housing_units',
    'Housing.Median Value of Owner-Occupied Units':     'housing_median_value',
    'Housing.Persons per Household':                    'housing_persons_per_hh',
    'Income.Median Houseold Income':                    'income_median_household',
    'Income.Per Capita Income':                         'income_per_capita',
    'Miscellaneous.Foreign Born':                       'misc_foreign_born_pct',
    'Miscellaneous.Land Area':                          'misc_land_area_sqmi',
    'Miscellaneous.Language Other than English at Home':'misc_lang_noneng_pct',
    'Miscellaneous.Living in Same House +1 Years':      'misc_same_house_1yr_pct',
    'Miscellaneous.Manufacturers Shipments':            'misc_manuf_shipments',
    'Miscellaneous.Mean Travel Time to Work':           'misc_mean_travel_time_min',
    'Miscellaneous.Percent Female':                     'misc_pct_female',
    'Miscellaneous.Veterans':                           'misc_veterans',
    'Population.2020 Population':                       'population_2020',
    'Population.2010 Population':                       'population_2010',
    'Population.Population per Square Mile':            'population_density',
    'Sales.Accommodation and Food Services Sales':      'sales_food_services',
    'Sales.Retail Sales':                               'sales_retail',
    'Employment.Firms.Total':                           'firms_total',
    'Employment.Firms.Women-Owned':                     'firms_women_owned',
    'Employment.Firms.Men-Owned':                       'firms_men_owned',
    'Employment.Firms.Minority-Owned':                  'firms_minority_owned',
    'Employment.Firms.Nonminority-Owned':               'firms_nonminority_owned',
    'Employment.Firms.Veteran-Owned':                   'firms_veteran_owned',
    'Employment.Firms.Nonveteran-Owned':                'firms_nonveteran_owned',
}
# Hay un tab en el nombre de una columna; limpiar primero
dem.columns = [c.replace('\t', '\t ') for c in dem.columns]
dem.rename(columns=rename_map, inplace=True)
print('Columnas sin mapear:', [c for c in dem.columns if c not in rename_map.values()])

In [ ]:
# --- -1 es centinela de dato suprimido (confidencialidad Census Bureau) ---
sentinel_cols = [
    'employment_nonemployer_estab', 'housing_units', 'housing_median_value',
    'misc_manuf_shipments', 'misc_veterans', 'sales_food_services', 'sales_retail'
]
for col in sentinel_cols:
    if col in dem.columns:
        mask = dem[col] == -1
        print(f'{col}: {mask.sum()} suprimidos')
        dem.loc[mask, col] = np.nan

In [ ]:
# --- Imputación por mediana estatal ---
# Razón: los datos suprimidos son por confidencialidad (condados pequeños),
# la mediana del estado es la mejor proxy disponible sin datos externos.
numeric_cols = dem.select_dtypes(include='number').columns.tolist()
# Extraer estado del campo county_name: "Abbeville County, SC" o campo state_name
# En este CSV el estado está en la columna state_name

before_nulls = dem[numeric_cols].isna().sum().sum()
dem[numeric_cols] = dem.groupby('state_name')[numeric_cols].transform(
    lambda s: s.fillna(s.median())
)
after_nulls = dem[numeric_cols].isna().sum().sum()
print(f'Nulos imputados por mediana estatal: {before_nulls - after_nulls}')

# Condados sin par estatal (ej. DC): imputar con mediana nacional
remaining = dem[numeric_cols].isna().sum().sum()
if remaining > 0:
    dem[numeric_cols] = dem[numeric_cols].fillna(dem[numeric_cols].median())
    print(f'Nulos imputados por mediana nacional: {remaining}')

print('Nulos finales:', dem.isnull().sum().sum())

In [ ]:
# --- Normalizar county_name para facilitar el join con otras tablas ---
# Quitar sufijos: " County", " Parish", " Borough", " Census Area", etc.
suffixes = r'\s+(County|Parish|Borough|Municipality|Census Area|City and Borough|City|Town|Village|District)$'
dem['county_clean'] = dem['county_name'].str.replace(suffixes, '', regex=True, flags=re.IGNORECASE).str.strip()

print(dem[['county_name', 'county_clean', 'state_name']].head(5))

In [ ]:
dem.to_csv(f'{OUT}/demographics_clean.csv', index=False)
print('Guardado demographics_clean.csv -', dem.shape)

---
## 3. Education data

In [ ]:
edu = pd.read_csv(f'{BASE}/education.csv', encoding='utf-8-sig', dtype={'FIPS Code': str})
print(edu.shape)
edu.head(2)

In [ ]:
# --- Drop filas de agregados nacionales y estatales (FIPS termina en 000 o es US) ---
edu['FIPS Code'] = edu['FIPS Code'].str.strip().str.zfill(5)
is_aggregate = edu['FIPS Code'].str.endswith('000') | (edu['Area name'] == 'United States')
print(f'Filas agregadas eliminadas: {is_aggregate.sum()}')
edu = edu[~is_aggregate].copy()
print('Filas condado restantes:', len(edu))

In [ ]:
# --- Separar clasificación urbana/rural ---
urban_cols = [
    'FIPS Code', 'State', 'Area name',
    '2003 Rural-urban Continuum Code', '2003 Urban Influence Code',
    '2013 Rural-urban Continuum Code', '2013 Urban Influence Code',
    'City/Suburb/Town/Rural 2013'
]
urban = edu[urban_cols].copy()
urban.columns = [
    'fips', 'state_abbr', 'area_name',
    'rucc_2003', 'uic_2003',
    'rucc_2013', 'uic_2013',
    'urban_category_2013'
]

# Convertir a numérico
for c in ['rucc_2003','uic_2003','rucc_2013','uic_2013']:
    urban[c] = pd.to_numeric(urban[c], errors='coerce')

# Imputar nulos (9 condados sin código) con la moda del estado
for c in ['rucc_2003','uic_2003','rucc_2013','uic_2013']:
    urban[c] = urban.groupby('state_abbr')[c].transform(
        lambda s: s.fillna(s.mode().iloc[0] if not s.mode().empty else np.nan)
    )
# Restantes -> moda nacional
for c in ['rucc_2003','uic_2003','rucc_2013','uic_2013']:
    urban[c] = urban[c].fillna(urban[c].mode().iloc[0])
urban['urban_category_2013'] = urban['urban_category_2013'].fillna('Unknown')

def rucc_label(code):
    try:
        c = int(float(code))
    except (ValueError, TypeError):
        return 'Unknown'
    if c == 1:   return 'Metro Large'
    elif c == 2: return 'Metro Medium'
    elif c == 3: return 'Metro Small'
    elif c <= 5: return 'Nonmetro Adjacent'
    elif c <= 7: return 'Nonmetro Nonadjacent'
    else:        return 'Rural Remote'

urban['rucc_2013_label'] = urban['rucc_2013'].apply(rucc_label)
print(urban['rucc_2013_label'].value_counts())
print('Nulls urban:', urban.isnull().sum().sum())
urban.head(3)

In [ ]:
# --- Limpiar columnas de educación ---
# Los conteos tienen comas como separadores de miles ("5,272" -> 5272)
# Los porcentajes son numéricos directos

def clean_edu_num(val):
    if pd.isna(val) or str(val).strip() == '':
        return np.nan
    return float(str(val).replace(',', '').strip())

edu_data_cols = [c for c in edu.columns if c not in urban_cols]

# Años y niveles disponibles
years  = ['1970', '1980', '1990', '2000', '2015-19']
levels = {
    'less_than_hs':          ['Less than a high school diploma', 'Less than a high school diploma'],
    'hs_only':               ['High school diploma only'],
    'some_college':          ['Some college (1-3 years)', "Some college or associate's degree"],
    'bachelors_or_higher':   ['Four years of college or higher', "Bachelor's degree or higher"],
}

In [ ]:
# --- Construir tabla larga: (fips, year, level) -> (count, pct) ---
# Matching case-insensitive porque los nombres de columna en el CSV usan
# "less than" (minúscula) mientras las variantes del dict usan "Less than".
cols_lower = {c.lower(): c for c in edu.columns}

records = []
for _, row in edu.iterrows():
    for yr in years:
        for level_code, col_variants in levels.items():
            cnt_col = next(
                (cols_lower[k] for k in cols_lower
                 if any(v.lower() in k for v in col_variants) and yr in k and 'percent' not in k),
                None
            )
            pct_col = next(
                (cols_lower[k] for k in cols_lower
                 if any(v.lower() in k for v in col_variants) and yr in k and 'percent' in k),
                None
            )
            records.append({
                'fips':        row['FIPS Code'],
                'state_abbr':  row['State'],
                'area_name':   row['Area name'],
                'period':      yr,
                'level_code':  level_code,
                'adults_count': clean_edu_num(row[cnt_col]) if cnt_col else np.nan,
                'adults_pct':   clean_edu_num(row[pct_col]) if pct_col else np.nan,
            })

edu_long = pd.DataFrame(records)
print('edu_long shape:', edu_long.shape)
print('Nulos iniciales:', edu_long[['adults_count','adults_pct']].isna().sum().to_dict())
edu_long.head(8)

In [ ]:
# --- Imputar NaN en datos históricos ---
# Estrategia: interpolación lineal por condado y nivel a través del tiempo,
# luego mediana del grupo (level_code + period) para los que no tienen suficientes puntos.

# Mapear periodos a numérico para interpolación
period_order = {'1970': 1970, '1980': 1980, '1990': 1990, '2000': 2000, '2015-19': 2017}
edu_long['period_num'] = edu_long['period'].map(period_order)

before = edu_long[['adults_count','adults_pct']].isna().sum()

# Interpolar dentro de cada (fips, level_code)
edu_long = edu_long.sort_values(['fips', 'level_code', 'period_num'])
for col in ['adults_count', 'adults_pct']:
    edu_long[col] = edu_long.groupby(['fips', 'level_code'])[col].transform(
        lambda s: s.interpolate(method='linear', limit_direction='both')
    )

# Restantes -> mediana por (period, level_code)
for col in ['adults_count', 'adults_pct']:
    edu_long[col] = edu_long.groupby(['period', 'level_code'])[col].transform(
        lambda s: s.fillna(s.median())
    )

after = edu_long[['adults_count','adults_pct']].isna().sum()
print('Nulos antes:', before.to_dict())
print('Nulos después:', after.to_dict())

# adults_count debe ser entero
edu_long['adults_count'] = edu_long['adults_count'].round(0).astype('Int64')

In [ ]:
# Validación: porcentajes dentro de rango
out_of_range = edu_long[(edu_long['adults_pct'] < 0) | (edu_long['adults_pct'] > 100)]
print('Porcentajes fuera de rango:', len(out_of_range))
edu_long['adults_pct'] = edu_long['adults_pct'].clip(0, 100)

edu_long.to_csv(f'{OUT}/education_clean.csv', index=False)
urban.to_csv(f'{OUT}/urban_class_clean.csv', index=False)
print('Guardado education_clean.csv -', edu_long.shape)
print('Guardado urban_class_clean.csv -', urban.shape)

---
## 4. Population Density data

In [ ]:
pop = pd.read_csv(f'{BASE}/PopulationDensityData.csv', encoding='utf-8-sig', dtype={'FIPS Code': str})
print(pop.shape)
pop.head(3)

In [ ]:
pop.columns = ['county_name', 'state_name', 'fips', 'population', 'area_sqmi', 'density_per_sqmi']

# --- Normalizar FIPS ---
pop['fips'] = pop['fips'].str.strip()

# Fila problemática: FIPS = '02-' (Alaska Unorganized Borough)
bad_fips = pop[~pop['fips'].str.match(r'^\d{5}$')]
print('Filas con FIPS inválido:')
print(bad_fips[['county_name','state_name','fips']])

In [ ]:
# Alaska Unorganized Borough -> FIPS oficial es 02270 (pero fue abolido en 2015)
# Lo mapeamos y aceptamos; si no cruza con elections simplemente quedará sin votos
pop.loc[pop['fips'] == '02-', 'fips'] = '02270'

# Verificar que todos son 5 dígitos
pop['fips'] = pop['fips'].str.zfill(5)
print('FIPS lengths:', pop['fips'].str.len().unique())

# Validar tipos numéricos
for c in ['population', 'area_sqmi', 'density_per_sqmi']:
    pop[c] = pd.to_numeric(pop[c], errors='coerce')

print('Nulls:', pop.isnull().sum())

In [ ]:
# --- Recalcular densidad para consistencia (density = population / area) ---
mask_bad_density = pop['area_sqmi'] > 0
pop.loc[mask_bad_density, 'density_per_sqmi'] = (
    pop.loc[mask_bad_density, 'population'] / pop.loc[mask_bad_density, 'area_sqmi']
).round(2)

# Log-densidad: útil para modelos y visualizaciones (distribución muy sesgada)
pop['log_density'] = np.log1p(pop['density_per_sqmi']).round(4)

pop.describe()

In [ ]:
pop.to_csv(f'{OUT}/population_density_clean.csv', index=False)
print('Guardado population_density_clean.csv -', pop.shape)

---
## 5. Religion data

In [ ]:
rel = pd.read_csv(f'{BASE}/religion_data.csv', encoding='latin-1', dtype={'fips': str})
print(rel.shape)
rel.head(3)

In [ ]:
# --- Drop filas con FIPS vacío o que sean filas de total/resumen ---
bad = (
    rel["fips"].isna() |
    (~rel["fips"].str.strip().str.match(r"^\d{5}$", na=False))  # cubre 'Total', '', NaN
)
print(f"Filas inválidas eliminadas: {bad.sum()}")
rel = rel[~bad].copy()

# --- Zero-pad FIPS ---
rel["fips"] = rel["fips"].str.strip().str.zfill(5)
print("FIPS lengths:", rel["fips"].str.len().unique())

In [ ]:
# --- Quitar símbolo % de columnas de porcentaje y convertir a float ---
for col in ['adherents_pct_total_adherents', 'adherents_pct_total_population']:
    rel[col] = (
        rel[col]
        .astype(str)
        .str.replace('%', '', regex=False)
        .str.strip()
        .replace({'': np.nan, 'nan': np.nan})
        .astype(float)
    )

print('Rango pct_population:', rel['adherents_pct_total_population'].min(), '-', rel['adherents_pct_total_population'].max())

In [ ]:
# --- Adherentes: ~14 933 filas sin dato (suprimido por privacidad, congregaciones < umbral) ---
print('Adherentes nulos:', rel['adherents'].isna().sum())
print('Congregaciones cuando adherentes es nulo:', rel.loc[rel['adherents'].isna(), 'congregations'].describe())

In [ ]:
# --- Imputar adherentes: mediana por (group_code, state_name) ---
rel['adherents'] = pd.to_numeric(rel['adherents'], errors='coerce')
rel['congregations'] = pd.to_numeric(rel['congregations'], errors='coerce')

before = rel['adherents'].isna().sum()
rel['adherents'] = rel.groupby(['group_code', 'state_name'])['adherents'].transform(
    lambda s: s.fillna(s.median())
)
rel['adherents'] = rel.groupby('group_code')['adherents'].transform(
    lambda s: s.fillna(s.median())
)
# Grupos tan raros que no hay mediana disponible: 1 adherente por congregación como floor
mask_still = rel['adherents'].isna() & rel['congregations'].notna() & (rel['congregations'] > 0)
rel.loc[mask_still, 'adherents'] = rel.loc[mask_still, 'congregations']
rel['adherents'] = rel['adherents'].fillna(0).round(0).astype('Int64')

# congregaciones nulas -> 0 (no reportadas)
rel['congregations'] = rel['congregations'].fillna(0).astype('Int64')

after = rel['adherents'].isna().sum()
print(f'Adherentes imputados: {before - after}  |  restantes nulos: {after}')

In [ ]:
# --- Recalcular pct_total_adherents desde nulos usando total por condado ---
county_total_adherents = rel.groupby('fips')['adherents'].sum().rename('county_total_adherents')
rel = rel.join(county_total_adherents, on='fips')

mask_null_pct = rel['adherents_pct_total_adherents'].isna() & (rel['county_total_adherents'] > 0)
rel.loc[mask_null_pct, 'adherents_pct_total_adherents'] = (
    rel.loc[mask_null_pct, 'adherents'] / rel.loc[mask_null_pct, 'county_total_adherents'] * 100
).round(4)
rel.drop(columns=['county_total_adherents'], inplace=True)

# --- Calcular pct_total_population uniendo con datos de densidad poblacional ---
# Cargamos solo fips + population del CSV limpio de densidad
pop_lookup = pd.read_csv(
    f'{OUT}/population_density_clean.csv',
    usecols=['fips','population'],
    dtype={'fips': str}
)
rel = rel.merge(pop_lookup, on='fips', how='left')

mask_pop = rel['adherents_pct_total_population'].isna() & (rel['population'] > 0)
rel.loc[mask_pop, 'adherents_pct_total_population'] = (
    rel.loc[mask_pop, 'adherents'] / rel.loc[mask_pop, 'population'] * 100
).round(4)
rel.drop(columns=['population'], inplace=True)

# --- Adherentes aún nulos: grupos tan raros que no hay mediana disponible ---
# Usar 1 adherente por congregación como floor conservador
mask_still_null = rel['adherents'].isna() & rel['congregations'].notna() & (rel['congregations'] > 0)
rel.loc[mask_still_null, 'adherents'] = rel.loc[mask_still_null, 'congregations'].astype(int)
# Los que no tienen ni congregaciones: marcar como 0 (sin datos reportados)
rel['adherents'] = rel['adherents'].fillna(0)

# Validar rangos
rel['adherents_pct_total_adherents'] = rel['adherents_pct_total_adherents'].clip(0, 100)
rel['adherents_pct_total_population'] = rel['adherents_pct_total_population'].clip(0, 100)
# Remaining pct nulls: condados sin match de población -> 0
rel['adherents_pct_total_population'] = rel['adherents_pct_total_population'].fillna(0.0)
rel['adherents_pct_total_adherents']  = rel['adherents_pct_total_adherents'].fillna(0.0)

print('Nulls finales:')
print(rel.isnull().sum())

In [ ]:
# --- Normalizar group_name: quitar acentos problemáticos (encoding latin-1) ---
import unicodedata
def normalize_str(s):
    if pd.isna(s):
        return s
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii').strip()

rel['group_name'] = rel['group_name'].apply(normalize_str)
rel['county_name'] = rel['county_name'].apply(normalize_str)

rel.head(3)

In [ ]:
rel.to_csv(f'{OUT}/religion_clean.csv', index=False)
print('Guardado religion_clean.csv -', rel.shape)

---
## 6. Resumen de calidad final

In [ ]:
summary = {
    'elections':          el,
    'demographics':       dem,
    'education_long':     edu_long,
    'urban_class':        urban,
    'population_density': pop,
    'religion':           rel,
}

for name, df in summary.items():
    nulls = df.isnull().sum().sum()
    print(f'{name:<22}  rows={len(df):>7}  cols={len(df.columns):>3}  nulls={nulls:>6}')

In [ ]:
# --- Verificar cobertura FIPS entre datasets ---
fips_elections = set(el['fips'])
fips_density   = set(pop['fips'])
fips_religion  = set(rel['fips'].unique())
fips_edu       = set(edu_long['fips'].unique())

print(f'Elections FIPS:  {len(fips_elections)}')
print(f'Density FIPS:    {len(fips_density)}')
print(f'Religion FIPS:   {len(fips_religion)}')
print(f'Education FIPS:  {len(fips_edu)}')
print()
print(f'En elections pero no en density: {len(fips_elections - fips_density)}')
print(f'En elections pero no en edu:     {len(fips_elections - fips_edu)}')
print(f'En elections pero no en religion:{len(fips_elections - fips_religion)}')

In [24]:
import pandas as pd
import numpy as np
import re

BASE = 'backend/datasets'
OUT  = 'backend/datasets/cleaned'

import os
os.makedirs(OUT, exist_ok=True)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
dem = pd.read_csv(f'{BASE}/DemographicsData.csv', encoding='utf-8-sig')
len(dem.columns)

43

In [25]:
dem.columns

Index(['County', 'State', 'Age.Percent 65 and Older',
       'Age.Percent Under 18 Years', 'Age.Percent Under 5 Years',
       'Education.Bachelor's Degree or Higher',
       'Education.High School or Higher',
       'Employment.Nonemployer Establishments',
       'Ethnicities.American Indian and Alaska Native Alone',
       'Ethnicities.Asian Alone', 'Ethnicities.Black Alone',
       'Ethnicities.Hispanic or Latino',
       'Ethnicities.Native Hawaiian and Other Pacific Islander Alone',
       'Ethnicities.Two or More Races', 'Ethnicities.White Alone',
       'Ethnicities.White Alone\t not Hispanic or Latino',
       'Housing.Homeownership Rate', 'Housing.Households',
       'Housing.Housing Units', 'Housing.Median Value of Owner-Occupied Units',
       'Housing.Persons per Household', 'Income.Median Houseold Income',
       'Income.Per Capita Income', 'Miscellaneous.Foreign Born',
       'Miscellaneous.Land Area',
       'Miscellaneous.Language Other than English at Home',
       'Mi